<a href="https://colab.research.google.com/github/Heptazero/ml-course-labs/blob/main/deep-learning/exp1_cnn_alexnet_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 实验一：卷积神经网络实践（AlexNet 与 ResNet）

## 1. 环境部署与测试

In [ ]:
%matplotlib inline
import time
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils import data
from torchvision import transforms
from matplotlib import pyplot as plt

def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()
print('PyTorch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Fashion-MNIST 数据集与训练评估函数

In [ ]:
def get_fashion_mnist_labels(labels):
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
                   'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]

def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        if torch.is_tensor(img):
            ax.imshow(img.numpy(), cmap='gray')
        else:
            ax.imshow(img, cmap='gray')
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i], fontsize=9)
    plt.tight_layout()
    plt.show()
    return axes

def load_data_fashion_mnist(batch_size, resize=None):
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True, num_workers=2),
            data.DataLoader(mnist_test, batch_size, shuffle=False, num_workers=2))

sample_iter, _ = load_data_fashion_mnist(18)
X_sample, y_sample = next(iter(sample_iter))
print('Batch shape:', X_sample.shape, X_sample.dtype, y_sample.shape, y_sample.dtype)
show_images(X_sample[:, 0], 2, 9, titles=get_fashion_mnist_labels(y_sample))

In [ ]:
def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

def evaluate_accuracy_gpu(net, data_iter, device=None):
    if isinstance(net, nn.Module):
        net.eval()
        if not device:
            device = next(iter(net.parameters())).device
    correct, total = 0.0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            correct += accuracy(net(X), y)
            total += y.numel()
    return correct / total

def train_ch6(net, train_iter, test_iter, num_epochs, lr, device):
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    total_time = 0.0
    total_examples = 0

    for epoch in range(num_epochs):
        loss_sum, acc_sum, n = 0.0, 0.0, 0
        net.train()
        for X, y in train_iter:
            t0 = time.time()
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                loss_sum += float(l) * X.shape[0]
                acc_sum += accuracy(y_hat, y)
                n += X.shape[0]
            total_time += time.time() - t0
        train_l = loss_sum / n
        train_acc = acc_sum / n
        test_acc = evaluate_accuracy_gpu(net, test_iter, device)
        total_examples += n
        history['train_loss'].append(train_l)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        print(f'epoch {epoch + 1:2d}/{num_epochs}: '
              f'loss {train_l:.3f}, train acc {train_acc:.3f}, test acc {test_acc:.3f}')

    speed = total_examples / total_time
    history['speed'] = speed
    print(f'loss {history["train_loss"][-1]:.3f}, '
          f'train acc {history["train_acc"][-1]:.3f}, '
          f'test acc {history["test_acc"][-1]:.3f}')
    print(f'{speed:.1f} examples/sec on {str(device)}')

    epochs = range(1, num_epochs + 1)
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, history['train_loss'], '-', label='train loss')
    plt.plot(epochs, history['train_acc'], 'm--', label='train acc')
    plt.plot(epochs, history['test_acc'], 'g-.', label='test acc')
    plt.xlabel('epoch')
    plt.xlim(1, num_epochs)
    plt.legend()
    plt.grid()
    plt.show()
    return history

## 3. AlexNet（第 7.1 节）

### 3.1 模型配置与结构检查

In [ ]:
alexnet = nn.Sequential(
    nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Flatten(),
    nn.Linear(6400, 4096), nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(4096, 4096), nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(4096, 10)
)

X = torch.randn(1, 1, 224, 224)
for layer in alexnet:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t', X.shape)

alexnet_params = sum(p.numel() for p in alexnet.parameters())
print(f'AlexNet parameters: {alexnet_params:,}')

### 3.2 训练与测试 AlexNet

In [ ]:
batch_size = 128
train_iter_alex, test_iter_alex = load_data_fashion_mnist(batch_size, resize=224)
lr, num_epochs = 0.01, 10
alexnet_hist = train_ch6(alexnet, train_iter_alex, test_iter_alex, num_epochs, lr, device)

## 4. ResNet-18（第 7.6 节）

### 4.1 残差块与模型配置

In [ ]:
class Residual(nn.Module):
    def __init__(self, input_channels, num_channels,
                 use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels,
                               kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels,
                               kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

blk1 = Residual(3, 3)
X_blk = torch.rand(4, 3, 6, 6)
print('Residual(3, 3):', blk1(X_blk).shape)

blk2 = Residual(3, 6, use_1x1conv=True, strides=2)
print('Residual(3, 6, 1x1, s=2):', blk2(X_blk).shape)

In [ ]:
def resnet_block(input_channels, num_channels, num_residuals, first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels,
                                use_1x1conv=True, strides=2))
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk

b1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
                   nn.BatchNorm2d(64), nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))

resnet18 = nn.Sequential(b1, b2, b3, b4, b5,
                         nn.AdaptiveAvgPool2d((1, 1)),
                         nn.Flatten(), nn.Linear(512, 10))

X = torch.rand(size=(1, 1, 224, 224))
for layer in resnet18:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t', X.shape)

resnet_params = sum(p.numel() for p in resnet18.parameters())
print(f'ResNet-18 parameters: {resnet_params:,}')

### 4.2 训练与测试 ResNet-18

In [ ]:
lr, num_epochs, batch_size = 0.05, 10, 256
train_iter_res, test_iter_res = load_data_fashion_mnist(batch_size, resize=96)
resnet_hist = train_ch6(resnet18, train_iter_res, test_iter_res, num_epochs, lr, device)

## 5. 实验结果对比与预测可视化

In [ ]:
print(f'{"Model":<12} {"Params":>12} {"Train Loss":>12} {"Train Acc":>12} {"Test Acc":>12} {"Speed (ex/s)":>14}')
print('-' * 78)
print(f'{"AlexNet":<12} {alexnet_params:>12,} {alexnet_hist["train_loss"][-1]:>12.3f} '
      f'{alexnet_hist["train_acc"][-1]:>12.3f} {alexnet_hist["test_acc"][-1]:>12.3f} '
      f'{alexnet_hist["speed"]:>14.1f}')
print(f'{"ResNet-18":<12} {resnet_params:>12,} {resnet_hist["train_loss"][-1]:>12.3f} '
      f'{resnet_hist["train_acc"][-1]:>12.3f} {resnet_hist["test_acc"][-1]:>12.3f} '
      f'{resnet_hist["speed"]:>14.1f}')

epochs = range(1, len(alexnet_hist['test_acc']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, alexnet_hist['train_loss'], 'o-', label='AlexNet')
axes[0].plot(epochs, resnet_hist['train_loss'], 's-', label='ResNet-18')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('train loss')
axes[0].set_title('Train Loss')
axes[0].legend()
axes[0].grid()

axes[1].plot(epochs, alexnet_hist['test_acc'], 'o-', label='AlexNet')
axes[1].plot(epochs, resnet_hist['test_acc'], 's-', label='ResNet-18')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('test acc')
axes[1].set_title('Test Accuracy')
axes[1].legend()
axes[1].grid()
plt.tight_layout()
plt.show()

In [ ]:
resnet18.eval()
X_test, y_test = next(iter(test_iter_res))
with torch.no_grad():
    preds = resnet18(X_test[:12].to(device)).argmax(dim=1).cpu()
trues = get_fashion_mnist_labels(y_test[:12])
pred_labels = get_fashion_mnist_labels(preds)
titles = [f'{t}\n({p})' for t, p in zip(trues, pred_labels)]
show_images(X_test[:12, 0], 2, 6, titles=titles, scale=1.8)